# MS Capital: Four Experiments That Did Not Help

After the first release I kept asking the same question of the MSCapital pipeline: *is there signal it does not extract?* Four more experiments, each with its decision rule written **before** the run, each scored as a paired difference against the model it was meant to improve. None of them was shown to help. This notebook is the one-picture version; the write-up, code and tests are in the repository.

The public leaderboard score of the shipped ensemble stayed **0.129**.

## Validation correction — 21 October 2026

The historical ensemble cosine **+0.14088** was obtained by fitting blend weights and scoring them on the **same fold targets**. It is an optimistic fitted-blend diagnostic, not an unbiased out-of-sample estimate. Agreement with the heuristic period adjustment (+0.14084) does not validate either estimate independently.

The repository now fits blend weights only on **earlier folds**, freezes them before scoring the current fold, and excludes the first fold from ensemble scoring. Historical base-model early stopping used the scored validation fold, so those scores are not fully nested outer-test estimates. Existing results below remain historical and are not silently replaced.

The LightGBM holdout (+0.15171) and submitted v4 ensemble leaderboard result (+0.12900) concern different models and evaluation scopes. This notebook re-executes the published result analysis; it does **not** retrain the full competition pipeline or submit predictions.

**NEXUS applied data science · Senanur Çetin · Non-commercial research**

[Validation protocol](https://github.com/senanurcetin/ms-capital-market-forecasting/blob/main/docs/VALIDATION.md) · [Model card](https://github.com/senanurcetin/ms-capital-market-forecasting/blob/main/MODEL_CARD.md)


## What was measured

| Experiment | Paired gain in cosine | 95% interval | Better in |
|---|---:|---|---|
| Drop 28 near-duplicate features | -0.00048 | [-0.00218, +0.00121] | 4 of 6 folds x seeds |
| Learned sequence model (CNN) blended in | -0.00091 | [-0.01042, +0.00859] | 4 of 8 months |
| Regime from the feature space | -0.00101 | [-0.00216, +0.00015] | 9 of 23 months |
| Four extra ensemble members | +0.00169 | [-0.00110, +0.00448] | 11 of 23 months |

A positive number means the change helped. Every interval includes zero, so none of the four counts as an improvement under the rule fixed in advance (the interval's lower bound above zero). The **widths** differ a lot and are part of the result: the regime experiment's interval is narrow enough to rule out a gain larger than about +0.0002, while the sequence-model interval (eight months) could not have detected a gain of +0.004.

These historical intervals use experiment-specific units and assumptions. They are exploratory, are not adjusted for multiple comparisons, and do not establish equivalence or absence of signal. Fold/seed measurements and adjacent months can be dependent. No new experiment is claimed by re-rendering this figure.

In [ ]:
import matplotlib.pyplot as plt

# (label, paired gain, interval low, interval high, note) - read from results/*.json in the repository
rows = [('Drop 28 near-duplicate features', -0.00048488091551775653, -0.0021752984541391894, 0.0012055366231036762, '4 of 6 folds x seeds'), ('Learned sequence model (CNN) blended in', -0.0009110539104981493, -0.010416483871854183, 0.008594376050857885, '4 of 8 months'), ('Regime from the feature space', -0.001005702556158474, -0.0021601469777198026, 0.00014874186540285482, '9 of 23 months'), ('Four extra ensemble members', 0.00169214799802958, -0.0010976288978676973, 0.004481924893926857, '11 of 23 months')]
INK, MUTED, MARK = "#1f2937", "#6b7280", "#1d4ed8"

fig, ax = plt.subplots(figsize=(9.5, 3.9))
ys = list(range(len(rows)))[::-1]
for y, (name, m, lo, hi, _) in zip(ys, rows):
    ax.plot([lo, hi], [y, y], color=MARK, lw=2, solid_capstyle="round")
    ax.plot([m], [y], "o", color=MARK, ms=8, mec="white", mew=2)
    ax.text(0.0125, y, f"{m:+.4f}  [{lo:+.4f}, {hi:+.4f}]", va="center", ha="left", color=INK, fontsize=9)
ax.axvline(0, color=MUTED, lw=1)
ax.axvline(0.0041, color=MUTED, lw=1, ls=":")
ax.set_ylim(-0.75, len(rows) - 0.4)
ax.text(0.0043, -0.62, "0.0041: fold-to-fold std", color=MUTED, fontsize=8, ha="left", va="center")
ax.set_yticks(ys)
ax.set_yticklabels([r[0] for r in rows], color=INK, fontsize=10)
ax.set_xlim(-0.0115, 0.0115)
ax.set_xlabel("paired gain in cosine (right of zero = helps)", color=MUTED)
ax.set_title("Four experiments, none shown to help", loc="left", color=INK, fontsize=12)
ax.tick_params(colors=MUTED)
ax.grid(axis="x", color="#e5e7eb", lw=0.8)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color("#d1d5db")
fig.subplots_adjust(left=0.38, right=0.77, top=0.86, bottom=0.17)
plt.show()

## How to read it

- **Not shown is not falsified.** Where an interval is wide, the experiment could not separate a small gain from none.
- **Sequence model.** A small dilated CNN over the 600 s market window scores 0.077 alone (the hand-built path statistics scored 0.0057), and its predictions correlate only 0.54 with the tabular model's, so it carries signal of its own. Blending it in was not shown to add to the tabular model.
- **Regime.** Sample order carries no information in the test set (block-mean autocorrelation +0.01 against +0.93 in train, and -0.05 within a month in train), so anything built from neighbouring samples cannot be computed at scoring time. Estimating the regime from the feature space instead changed nothing.
- **Extra ensemble members.** The monthly mean is positive, but the pooled cosine the competition scores is slightly *lower* for the seven-member blend (0.14350 against 0.14383), and the fitted weights follow validation rank, not test.

## Links

- Code, tests and write-up: https://github.com/senanurcetin/ms-capital-market-forecasting
- Release v1.1.0: https://github.com/senanurcetin/ms-capital-market-forecasting/releases/tag/v1.1.0
- Model card and files: https://huggingface.co/senanurcetin/ms-capital-market-forecasting
- Live dashboard: https://ms-capital-market-forecasting-mfy6rngulq4fpaovzrhntf.streamlit.app/
- Earlier notebook: https://www.kaggle.com/code/senanuretin/ms-capital-my-hold-out-was-a-lucky-stretch
- Competition: https://www.kaggle.com/competitions/ms-capital-real-financial-market-forecasting

*Research only. Nothing here is investment advice, and the competition data is not redistributed.*